# Week 7

This week, we're covering [`numpy`](https://numpy.org/) (which stands for *numerical Python*). This package is the **foundation of most other data science packages**. It is very fast due to using vectorization, contiguous arrays, and C memory management under the hood.

Run the code below to import `numpy` and test the speed of operations using nested lists compared to `ndarrays`. By convention, `numpy` is imported `as np`, so all functions are accessible by `np.function_name`.

We're also importing `matplotlib` to show plots, but this package will be covered in later weeks.

In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(seed=42) #for random number generation

In [3]:
n = 3000
matrix0 = [[random.random() for _ in range(n)] for _ in range(n)]
matrix1 = [[random.random() for _ in range(n)] for _ in range(n)]

In [ ]:
%%time
sum_matrix = []
for row in range(n):
  row_sum = []
  for col in range(n):
      row_sum.append(matrix0[row][col] + matrix1[row][col])
  sum_matrix.append(row_sum)

CPU times: user 2.28 s, sys: 138 ms, total: 2.42 s
Wall time: 2.42 s


In [ ]:
%%time
arr0 = np.array(matrix0)
arr1 = np.array(matrix1)
sum_arr = arr0 + arr1

CPU times: user 918 ms, sys: 178 ms, total: 1.1 s
Wall time: 1.09 s


A standard Python list is not memory-efficient for storing a lot of data. Numpy arrays (`ndarray`, *n-dimensional array*) are homogeneous and stored in contiguous memory blocks. They can be obtained by several ways, including conversion from list:

In [ ]:
# 1D array
arr_1d = np.array([12.5, 13.0, 11.8, 14.2, 12.1])

# 2D array
arr_2d = np.array([
    [0.1, 0.2, 0.1, 0.3],
    [0.9, 0.8, 0.7, 0.9],
    [1.5, 1.4, 1.6, 1.5]
])

print(f"array dimensions: {arr_2d.ndim}")
print(f"array shape: {arr_2d.shape}")
print(f"total number of values inside: {arr_2d.size}")

array dimensions: 2
array shape: (3, 4)
total number of values inside: 12


"Blank" or template-based arrays can also be easily created:

In [ ]:
arr_linear = np.arange(0, 120, 10) # 0 to 110 in steps of 10
arr_zeros = np.zeros((3, 4))        # 3x4 array of zeros
arr_ones = np.ones(12)          # 1D array of ones

Numpy uses indexing without iteration. Let's say that our 2d array as a microplate and try selections:

In [ ]:
microplate = arr_2d.copy()  #copy prevents modifying the original array, as normally view is returned.

row_two = microplate[1, :]
top_right_quadrant = microplate[0:2, 2:4] #selecting specific parts

high_signal_mask = microplate > 1.0 # find all wells where value was more than 1.0
high_signals = microplate[high_signal_mask]

mid_signals = microplate[(microplate > 0.5) & (microplate < 1.0)] #getting medium values

cleaned_plate = microplate.copy()
cleaned_plate[cleaned_plate < 0.2] = 0.0  #zeroing low values

Operations on arrays are applied element-wise. Numpy uses broadcasting to try to match dimensions. When aggregating (summing, mean-ing, etc.) data, the `axis` parameter defines the direction of the operation.

In [ ]:
# multiplication and subtraction on each element
adjusted_plate = microplate * 1.5 - 0.05

# aggregation of rows
column_means = microplate.mean(axis=0)

#aggregation of columnms
row_maxes = microplate.max(axis=1)

#broadcasting: 1d noise array can be subtracted from 2d signal array
background_noise = np.array([0.05, 0.05, 0.02, 0.1])
true_signal = microplate - background_noise # 1D array broadcasts automatically across the rows

**Refer to the slides, [numpy documentation](https://numpy.org/doc/2.5/reference/index.html#reference), or google for more functions!**

---

## Example

We have a batch of 5 replicates, each containing a sequence of 100 continuous sensor readouts. The data is noisy. We need to:
1. Generate the synthetic data matrix
2. Calculate the mean squared error (MSE) between our replicates and an ideal theoretical baseline. $MSE = (1/n) * sum(square(predictions - labels))$
3. Identify and isolate the exact timestamps where the biological signal spikes above the mean + standard deviation
4. Reshape the isolated spikes into two columns for downstream processing

In [ ]:
# generate data
raw_sensor_data = rng.random((5, 100)) * 50  # values between 0 and 50

# we will calculate MSE against an ideal baseline of 25.0
ideal_baseline = np.ones(100) * 25.0

# we can apply MSE across axis = 1 to get the MSE for each replicate
mse_per_replicate = (1/100) * np.sum((raw_sensor_data - ideal_baseline) ** 2, axis=1)
print(f"MSE per replicate: {mse_per_replicate}")

# let's calculate means for masking, so they will adapt to the data
global_mean = raw_sensor_data.mean()
global_std = np.std(raw_sensor_data)
#set threshold to mean+std
threshold = global_mean + global_std

# find indices where condition is true using nonzero()
# it returns a tuple of arrays: (row_indices, col_indices)
spike_coordinates = np.nonzero(raw_sensor_data > threshold)

# extract actual values
spike_values = raw_sensor_data[raw_sensor_data > threshold]

# if we have an even number of spikes, reshape into a 2 column format
if spike_values.size % 2 == 0:
    reshaped_spikes = spike_values.reshape(-1, 2)
else:
    # drop the last element to make it even, then reshape
    reshaped_spikes = spike_values[:-1].reshape(-1, 2)

print(f"reshaped spikes array shape: {reshaped_spikes.shape}")

MSE per replicate: [209.50887639 219.99080504 230.68332157 214.66112906 222.6266101 ]
reshaped spikes array shape: (54, 2)


## Exercises
Try to complete as much as you can.
Don't use `for` loops unless absolutely necessary!

### Exercise 1
Create a 3D array representing 3 days of experiments, where each day contains a 4x6 microplate of random float values between 0 and 1.
Extract a 2D slice representing the top-right 2x2 quadrant of the microplate *across all 3 days*. What is the `.shape` of your resulting slice?

In [ ]:
#your code here

### Exercise 2
Generate a 1D array of 50 random integers between 0 and 100. Find all values that are greater than the array's mean, but less than the array's maximum value minus 10. Replace all values meeting this condition with `-1`.

In [ ]:
#your code here

### Exercise 3
Create a 2D array of shape `(10, 5)` representing 10 patients and 5 biomarker readings. Create a 1D array of shape `(10,)` representing a baseline weight for each patient. Attempting to divide the 2D array by the 1D array will crash.
Fix the crash by using `np.newaxis` (or `np.expand_dims`) to properly align the dimensions so that each patient's 5 biomarkers are divided by their specific weight.

In [ ]:
#your code here

### Exercise 4
Write a Python function `safe_filter(matrix)` that takes a 2D array. Inside the function, slice the first two rows and columns. Multiply this slice by 100.
Write the function in such a way that the *original* matrix passed into the function remains completely unchanged in memory. Prove it by printing the original array before and after calling the function.

In [ ]:
#your code here

### Exercise 5
Generate a 2D array of shape `(8, 8)` containing sequential numbers from 1 to 64 (using `np.arange` and `reshape`). Transpose the matrix. Then, flip the matrix *only* along the rows (axis=0). Finally, flatten the matrix back into a 1D array using `.ravel()`.

In [ ]:
#your code here

### Exercise 6
Generate two separate 1D arrays of size 1000, `array_A` and `array_B`, containing random floats. Calculate the Euclidean distance between these two arrays entirely without loops.
*Euclidean distance is the square root of the sum of the squared differences between corresponding elements.*

In [ ]:
#your code here

### Exercise 7
Create two 2D arrays: `batch_1` with shape `(4, 3)` and `batch_2` with shape `(4, 3)`. Concatenate them vertically to create an `(8, 3)` array. Then, concatenate them horizontally to create a `(4, 6)` array. Finally, take the `(4, 6)` array and use `np.hsplit()` to split it into three equal `(4, 2)` arrays.

### Exercise 8
Create a 2D array of shape `(10, 10)` with random integers between 0 and 50.
Using `np.nonzero()`, find the exact `(row, column)` index pairs where the value is exactly equal to the minimum value of the entire array. Format your output as a list of Python tuples.

### Exercise 9
Write a custom Python function `max_min_spread(data, axis)` that accepts an N-dimensional array and an axis integer. The function should compute the maximum value along the given axis, subtract the minimum value along that same axis, and return the resulting array. Test it on a `(5, 10)` array using `axis=0` and `axis=1`.



### Exercise 10
1. Use `rng.random()` to generate a `(500, 4)` matrix representing 500 samples with 4 features each.
2. Calculate the sum of features for each sample (axis=1).
3. Create a boolean mask to keep only the samples where the sum of features is greater than 2.0.
4. Apply the mask to filter the original matrix.
5. Save the filtered matrix to a file named `filtered_samples.csv` using `np.savetxt()` (ensure it is saved as comma-separated).
6. Load the file back into a new variable using `np.loadtxt()` to verify it worked.